
# SFT Training Model

## Goal
Create a presentable notebook for supervised fine-tuning that:
- loads a training CSV with id, prompt, and answer
- converts the rows into chat-style training samples
- fine-tunes Qwen/Qwen2.5-1.5B-Instruct with LoRA on an 8-bit base model
- saves the final adapter and tokenizer
- runs one short post-training generation test

## Approach
- a local instruction-tuned language model: Qwen/Qwen2.5-1.5B-Instruct
- 8-bit quantization to reduce GPU memory usage
- LoRA to train only a small adapter instead of the full base model
- a simple train/validation split for basic evaluation during training


In [ ]:
# Import libraries
import os
import csv
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig


## 1. Configuration

Here the static parameters for supervised fine-tuning are defined.
This includes the model, data paths, LoRA settings, and training settings.


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
csv_path = "./data/dataset_sfttraining_total.csv"

output_dir = "./sft_out/qwen25_tax_sft_8bit"
final_adapter_dir = "./sft_out/qwen25_tax_sft_8bit/final_adapter"

validation_share = 0.1
random_seed = 42
max_length = 768

num_epochs = 3
learning_rate = 2e-4
train_batch_size = 1
eval_batch_size = 1
gradient_accumulation_steps = 8
logging_steps = 10

lora_r = 16
lora_alpha = 32
lora_dropout = 0.05

system_prompt = (
    "You are a careful Austrian tax law assistant. "
    "Answer briefly, factually, and clearly in one plain-text paragraph. "
    "Do not invent legal references.")

test_prompt = "Wann liegt der Leistungsort einer sonstigen Leistung in Österreich?"


## 2. GPU Configuration

Training should only start if CUDA is available.
This notebook also enables TF32 where possible and automatically picks bf16 if the GPU supports it.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This script needs a GPU.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Torch version:", torch.__version__)
print("Using bf16:", use_bf16)
print()


## 3. Load the CSV

This section reads the CSV file.
It also handles the broken export case where the full row was stored inside the first column.


In [ ]:
if not os.path.exists(csv_path):
    raise ValueError(f"CSV file not found: {csv_path}")

rows = []

with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=",", quotechar='"')

    if reader.fieldnames is None:
        raise ValueError("CSV header could not be read.")

    needed_columns = ["id", "prompt", "answer"]
    for col in needed_columns:
        if col not in reader.fieldnames:
            raise ValueError(
                f"Missing required column: {col}. Found columns: {reader.fieldnames}")

    for row in reader:
        # Normal case:
        # The CSV already has separate id, prompt, and answer fields.
        if row["prompt"] is not None and row["answer"] is not None:
            current_id = str(row["id"]).strip().strip('"')
            current_prompt = str(row["prompt"]).strip()
            current_answer = str(row["answer"]).strip()

            if current_id != "" and current_prompt != "" and current_answer != "":
                rows.append(
                    {"id": current_id, "prompt": current_prompt, "answer": current_answer})
            continue

        # Broken export case:
        # The full row is stored as one string inside the id column.
        raw_line = row["id"]

        if raw_line is None:
            continue

        parsed = next(csv.reader([raw_line], delimiter=",", quotechar='"'), None)

        if parsed is None or len(parsed) < 3:
            continue

        current_id = str(parsed[0]).strip().strip('"')
        current_prompt = str(parsed[1]).strip()
        current_answer = str(parsed[2]).strip()

        if current_id != "" and current_prompt != "" and current_answer != "":
            rows.append(
                {"id": current_id, "prompt": current_prompt, "answer": current_answer})

print("Usable rows:", len(rows))
print()

if len(rows) == 0:
    raise ValueError("No usable rows in CSV")

if len(rows) < 2:
    raise ValueError("Min. 2 usable rows")


## 4. Build the Training Dataset

The CSV rows are converted into chat-style examples.
This matches the instruction format of the model more closely than plain text pairs.


In [ ]:
examples = []

for row in rows:
    examples.append(
        {"id": row["id"],
            "prompt": [{"role": "system", "content": system_prompt},
                {"role": "user", "content": row["prompt"]}], "completion": [
                {"role": "assistant", "content": row["answer"]}]})

dataset = Dataset.from_list(examples)

print("Dataset size:", len(dataset))
print()


## 5. Train / Validation Split

The dataset is split into a training part and a validation part.
At least one example is always kept for validation.


In [ ]:
validation_size = max(1, int(len(dataset) * validation_share))

if validation_size >= len(dataset):
    validation_size = 1

split_dataset = dataset.train_test_split(test_size=validation_size, seed=random_seed, shuffle=True)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))
print()


## 6. Load the Tokenizer

The tokenizer converts text into tokens.
Padding is set to the right, and the EOS token is reused as padding if needed.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print()


## 7. Load the Base Model

The base model is loaded in **8-bit mode** to reduce memory usage.
Caching is disabled because training works better without it together with gradient checkpointing.


In [ ]:
quant_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map="auto", dtype=compute_dtype)

# Turn off cache during training
# This works better with gradient checkpointing
model.config.use_cache = False

print("Base model loaded.")
print()


## 8. Prepare the Model for LoRA Training

The quantized base model is prepared for low-bit training.
Then the LoRA adapter configuration is created.


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear")

print("LoRA config ready")
print()


## 9. Training Settings

This section defines the training arguments.


In [ ]:
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_epochs,
    learning_rate=learning_rate,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    logging_steps=logging_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=max_length,
    packing=False,
    gradient_checkpointing=True,
    fp16=not use_bf16,
    bf16=use_bf16,
    tf32=True,
    completion_only_loss=True,
    eos_token="<|im_end|>",
    dataset_num_proc=1)


## 10. Create the Trainer

The trainer handles tokenization, training, and evaluation.
This is the central object for the SFT run.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config)

print("Trainer created.")
print()


## 11. Start Training

This cell starts supervised fine-tuning.
Training logs and epoch evaluation outputs will appear below the cell during execution.


In [ ]:
print("Starting training...")
print()

trainer.train()

print()
print("Training finished.")
print()


## 12. Save the Result

After training, the adapter and tokenizer are saved to the final output folder.


In [ ]:
os.makedirs(final_adapter_dir, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print("Saved adapter to:", final_adapter_dir)
print()


## 13. Quick Test

A short generation test is run after training.
This checks whether the fine-tuned model can produce an answer for one sample tax-law prompt.


In [ ]:
trainer.model.eval()

test_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt}]

test_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True)

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=max_length
)

# Move the input tensors to the model device.
inputs = {key: value.to(trainer.model.device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("Test prompt:")
print(test_prompt)
print()
print("Model answer:")
print(generated_text)